# Análise do Titanic — SQL (SQLite) + Estatística em Python puro

**Objetivo:** explorar o dataset do Titanic usando **SQL** (SQLite) para consultar/agrupar os dados e **estatística descritiva implementada em Python puro** — média, mediana, moda, variância e desvio-padrão calculados na mão, sem usar `pandas.mean()` e afins.

> **Regra de ouro deste projeto:** o pandas serve só para **carregar e organizar** os dados (e depois **validar** os cálculos). As 5 métricas são implementadas manualmente, para entender como cada uma funciona por dentro.

## Seção 1 — Setup, carga dos dados e exploração inicial

Nesta seção vamos:
1. Importar as bibliotecas.
2. Ler o `titanic.csv` com pandas.
3. Criar um banco SQLite e carregar os dados nele (com `connection` e `cursor` explícitos).
4. Fazer uma exploração inicial via SQL.

### Conceitos rápidos
- **Por que SQLite e não só pandas?** O foco é treinar SQL num banco de verdade. SQLite é um banco completo dentro de **um único arquivo** `.db`, sem servidor para instalar.
- **O módulo `sqlite3` já vem com Python** (biblioteca padrão) — não precisa instalar nada.
- **`connection` x `cursor`:** a *connection* é o vínculo com o arquivo do banco e controla as transações (`commit`). O *cursor* é quem executa o SQL e percorre o resultado (`fetchone`/`fetchall`).

### 1.1 — Importar bibliotecas

- `pandas`: apenas para **ler e organizar** o CSV (NÃO para calcular as métricas).
- `sqlite3`: driver do banco SQLite (biblioteca padrão, já vem embutido).
- `os`: montar caminhos de arquivo de forma portável.

In [ ]:
import pandas as pd
import sqlite3
import os

print("pandas:", pd.__version__)
print("motor SQLite:", sqlite3.sqlite_version)

### 1.2 — Ler o `titanic.csv`

O notebook roda dentro da pasta `notebooks/`, então o CSV está em `../data/titanic.csv`.

O `pd.read_csv` já entende vírgulas dentro de aspas (importante no campo `Name`) e infere os tipos das colunas. Onde o valor está vazio, o pandas coloca **`NaN`** ("não é um número") — vamos tratar isso na hora de gravar no banco.

In [ ]:
CSV_PATH = os.path.join("..", "data", "titanic.csv")

df = pd.read_csv(CSV_PATH)

print("Dimensões (linhas, colunas):", df.shape)
print("Colunas:", list(df.columns))

# Espiar as primeiras linhas (df.head() renderiza uma tabela)
df.head()

### 1.3 — Criar o banco SQLite e carregar os dados

Aqui aparecem a `connection` e o `cursor` na prática:
1. `sqlite3.connect(...)` cria/abre o arquivo `../data/titanic.db` e devolve a **connection**.
2. `con.cursor()` cria o **cursor**, que executa os comandos.
3. `DROP TABLE IF EXISTS` + `CREATE TABLE`: definimos a tabela `passengers` com os tipos certos (`INTEGER`, `REAL`, `TEXT`). O `DROP` deixa a célula **reexecutável** sem erro de "tabela já existe".
4. `cur.executemany(...)` insere todas as linhas de uma vez com **parâmetros `?`** (forma segura: evita SQL injection e cuida do escape).
5. `con.commit()` **confirma a transação** — sem isso, nada é gravado de fato no arquivo.

**Detalhe importante (NaN → NULL):** o `sqlite3` só aceita `int`, `float`, `str`, `bytes` e `None` nativos do Python. O pandas entrega `NaN` para vazios e números como tipos do numpy. Por isso as funções `para_inteiro`/`para_decimal`/`para_texto` convertem cada valor para o tipo nativo e transformam `NaN` em `None` (que vira `NULL` no SQL).

> Repare na divisão de papéis: o **cursor** executa o SQL; a **connection** confirma (`commit`).

In [ ]:
DB_PATH = os.path.join("..", "data", "titanic.db")

# Conversores: tipo nativo do Python; NaN/None viram None (NULL no SQL)
def para_inteiro(valor):
    """Converte para int nativo; vazio (NaN/None) vira None (NULL)."""
    return None if pd.isna(valor) else int(valor)

def para_decimal(valor):
    """Converte para float nativo; vazio (NaN/None) vira None (NULL)."""
    return None if pd.isna(valor) else float(valor)

def para_texto(valor):
    """Converte para str; vazio (NaN/None) vira None (NULL)."""
    return None if pd.isna(valor) else str(valor)

# Montar a lista de registros já com os tipos tratados
registros = [
    (
        para_inteiro(linha.PassengerId), para_inteiro(linha.Survived), para_inteiro(linha.Pclass),
        para_texto(linha.Name), para_texto(linha.Sex), para_decimal(linha.Age),
        para_inteiro(linha.SibSp), para_inteiro(linha.Parch), para_texto(linha.Ticket),
        para_decimal(linha.Fare), para_texto(linha.Cabin), para_texto(linha.Embarked),
    )
    for linha in df.itertuples(index=False)
]

# 1) Conexao com o arquivo do banco
con = sqlite3.connect(DB_PATH)
# 2) Cursor para executar comandos
cur = con.cursor()

# 3) (Re)criar a tabela com os tipos corretos
cur.execute("DROP TABLE IF EXISTS passengers")
cur.execute("""
    CREATE TABLE passengers (
        PassengerId INTEGER PRIMARY KEY,
        Survived    INTEGER,
        Pclass      INTEGER,
        Name        TEXT,
        Sex         TEXT,
        Age         REAL,
        SibSp       INTEGER,
        Parch       INTEGER,
        Ticket      TEXT,
        Fare        REAL,
        Cabin       TEXT,
        Embarked    TEXT
    )
""")

# 4) Inserir todos os registros de uma vez
cur.executemany(
    "INSERT INTO passengers VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    registros,
)

# 5) Confirmar a transacao (sem isso, nada e gravado no arquivo)
con.commit()

print(f"Banco criado em: {DB_PATH}")
print(f"Registros inseridos: {len(registros)}")

### 1.4 — Exploração inicial (via SQL)

Agora usamos o cursor para fazer as primeiras perguntas ao banco:
- **Esquema** da tabela (`PRAGMA table_info`).
- **Total de linhas**.
- **Amostra** das 5 primeiras linhas.
- **Valores faltando** (`NULL`) nas colunas críticas.
- **Distribuição** das variáveis categóricas.

Tudo isso é SQL — ainda **sem calcular estatísticas**; isso virá nas próximas seções, em Python puro.

In [ ]:
print("=== Esquema da tabela 'passengers' ===")
for info_coluna in cur.execute("PRAGMA table_info(passengers)").fetchall():
    # info_coluna = (id, nome, tipo, notnull, default, pk)
    print(f"  {info_coluna[1]:<12} {info_coluna[2]}")

total = cur.execute("SELECT COUNT(*) FROM passengers").fetchone()[0]
print(f"\nTotal de passageiros: {total}")

In [ ]:
print("=== Amostra (5 primeiras linhas) ===")
for linha in cur.execute("SELECT PassengerId, Survived, Pclass, Name, Sex, Age, Fare FROM passengers LIMIT 5"):
    print(linha)

In [ ]:
print("=== Valores faltando (NULL) ===")
for coluna in ["Age", "Cabin", "Embarked", "Fare"]:
    faltando = cur.execute(f"SELECT COUNT(*) FROM passengers WHERE {coluna} IS NULL").fetchone()[0]
    print(f"  {coluna:<10}: {faltando} faltando ({faltando/total:.1%})")

In [ ]:
print("=== Distribuicoes (categoricas) ===")

print("\nSurvived (0 = morreu, 1 = sobreviveu):")
for valor, quantidade in cur.execute("SELECT Survived, COUNT(*) FROM passengers GROUP BY Survived"):
    print(f"  {valor}: {quantidade}")

print("\nPclass (classe do bilhete):")
for valor, quantidade in cur.execute("SELECT Pclass, COUNT(*) FROM passengers GROUP BY Pclass ORDER BY Pclass"):
    print(f"  {valor}: {quantidade}")

print("\nSex:")
for valor, quantidade in cur.execute("SELECT Sex, COUNT(*) FROM passengers GROUP BY Sex"):
    print(f"  {valor}: {quantidade}")

print("\nEmbarked (porto de embarque):")
for valor, quantidade in cur.execute("SELECT Embarked, COUNT(*) FROM passengers GROUP BY Embarked"):
    print(f"  {valor}: {quantidade}")

### Interpretação inicial

- **891 passageiros** no dataset. Dos quais **549 morreram (0)** e **342 sobreviveram (1)** — já dá pra sentir que a maioria não sobreviveu.
- **3ª classe domina** (491 pessoas), contra 216 na 1ª e 184 na 2ª — um navio socialmente desigual.
- **Mais homens (577) do que mulheres (314)**.
- **Dados faltando:** `Age` falta em ~20% (177) e `Cabin` em ~77% (687) — isso vai exigir cuidado nas análises de idade. `Embarked` só falta em 2 registros.

> **Fim da Seção 1.** A `connection` (`con`) e o `cursor` (`cur`) ficam abertos de propósito para reutilizarmos nas próximas seções (fecharemos com `con.close()` no fim do notebook).

## Seção 2 — Análises com SQL (8+ queries)

A partir daqui, cada query é uma subseção. Em cada uma: explico **o que ela responde**, mostro a **query formatada**, comento as **cláusulas** e **interpreto** o resultado.

> Para **exibir** o resultado como tabela uso `pd.read_sql_query(sql, con)`. Quem faz o trabalho é o **SQL rodando no SQLite** — o pandas aqui só busca e organiza a saída (uso permitido pelo CLAUDE.md).

### Query 1 — Taxa de sobrevivência geral

**Pergunta:** de todos os passageiros, qual a porcentagem que sobreviveu?

A coluna `Survived` vale `0` (morreu) ou `1` (sobreviveu). Por isso, **somar** a coluna (`SUM`) já dá o número de sobreviventes, e dividir pelo **total** (`COUNT`) dá a proporção.

**Cláusulas:**
- `COUNT(*)` — conta todas as linhas (total de passageiros).
- `SUM(Survived)` — soma os `1`s, ou seja, o número de sobreviventes.
- `100.0 * ... / ...` — o `100.0` (com casa decimal) força a **divisão decimal**; sem isso o SQLite faria divisão **inteira** e o resultado viria `0`.
- `ROUND(..., 2)` — arredonda para 2 casas decimais.
- `AS ...` — dá um **apelido** legível para cada coluna do resultado.

In [ ]:
consulta_1 = """
SELECT
    COUNT(*)                                   AS total_passageiros,
    SUM(Survived)                              AS total_sobreviventes,
    ROUND(100.0 * SUM(Survived) / COUNT(*), 2) AS taxa_sobrevivencia_pct
FROM passengers
"""

pd.read_sql_query(consulta_1, con)

**Interpretação:** dos **891 passageiros**, apenas **342 sobreviveram** — taxa de **38,38%**. Ou seja, **quase 2 em cada 3 pessoas morreram**.

Esse valor é a nossa **linha de base**. Nas próximas queries vamos quebrar por **classe** e **sexo** para descobrir *quem* tinha chance acima ou abaixo desses 38%.